In [1]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

print("Libraries loaded successfully!")

ModuleNotFoundError: No module named 'geopandas'

In [1]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
import geopandas as gpd

# Load building polygons
buildings = gpd.read_file(
r"C:\Users\Sohan\Desktop\project final year\Malabe_Buildings_Final.gpkg"
)

print(buildings.head())

print()
print("Number of buildings =", len(buildings))

   latitude  longitude  area_in_meters  confidence full_plus_code  \
0  6.901543  79.953005         88.5353      0.8634  6JRXWX23+J66J   
1  6.899889  79.957913         47.9530      0.7412  6JRXVXX5+X53C   
2  6.905843  79.946706        238.3709      0.8054  6JRXWW4W+8MPP   
3  6.896316  79.957827        171.7970      0.6709  6JRXVXW5+G4GM   
4  6.904015  79.958621         71.2049      0.6618  6JRXWX35+JC5Q   

                                            geometry  
0  MULTIPOLYGON (((79.95306 6.90152, 79.95302 6.9...  
1  MULTIPOLYGON (((79.95796 6.89985, 79.9579 6.89...  
2  MULTIPOLYGON (((79.94681 6.90583, 79.94676 6.9...  
3  MULTIPOLYGON (((79.95792 6.89634, 79.95788 6.8...  
4  MULTIPOLYGON (((79.95862 6.90411, 79.95859 6.9...  

Number of buildings = 19605


In [3]:
# Calculate total roof area

total_roof_area = buildings["area_in_meters"].sum()

print("Total rooftop area =", round(total_roof_area,2), "m²")

Total rooftop area = 2085028.06 m²


In [4]:
# Assumptions

solar_irradiance = 1700     # kWh/m²/year
panel_efficiency = 0.20     # 20%
performance_ratio = 0.75    # losses

annual_energy = (
    total_roof_area
    * solar_irradiance
    * panel_efficiency
    * performance_ratio
)

print("Annual Solar Energy =", round(annual_energy/1e6,2), "GWh/year")

Annual Solar Energy = 531.68 GWh/year


In [5]:
import geopandas as gpd

# Load buildings
buildings = gpd.read_file(
r"C:\Users\Sohan\Desktop\project final year\Malabe_Buildings_Final.gpkg"
)

# Load urban boundary
boundary = gpd.read_file(
r"C:\Users\Sohan\Desktop\project final year\Malabe_Urban_Area_Final.gpkg"
)

print("Buildings =", len(buildings))
print("Boundary polygons =", len(boundary))

Buildings = 19605
Boundary polygons = 1


In [6]:
# Convert both layers to UTM Zone 44N (meters)

buildings = buildings.to_crs(epsg=32644)
boundary = boundary.to_crs(epsg=32644)

print(buildings.crs)
print(boundary.crs)

EPSG:32644
EPSG:32644


In [7]:
from shapely.geometry import box

# Create a rectangle (1000 m × 1000 m)

xmin = 600000
ymin = 760000
xmax = 601000
ymax = 761000

user_polygon = box(xmin, ymin, xmax, ymax)

print(user_polygon)

POLYGON ((601000 760000, 601000 761000, 600000 761000, 600000 760000, 601000 760000))


In [8]:
selected_buildings = buildings[buildings.intersects(user_polygon)]

print("Buildings inside polygon =", len(selected_buildings))

Buildings inside polygon = 0


In [9]:
# Get boundary extent

xmin, ymin, xmax, ymax = boundary.total_bounds

print("xmin =", xmin)
print("ymin =", ymin)
print("xmax =", xmax)
print("ymax =", ymax)

xmin = 383355.96962058434
ymin = 761561.6216606162
xmax = 387561.3385026966
ymax = 764895.0666145973


In [10]:
# Calculate center of Malabe

center_x = (xmin + xmax)/2
center_y = (ymin + ymax)/2

print(center_x)
print(center_y)

385458.6540616405
763228.3441376068


In [11]:
from shapely.geometry import box

# 1 km × 1 km square
size = 500

user_polygon = box(
    center_x-size,
    center_y-size,
    center_x+size,
    center_y+size
)

print(user_polygon)

POLYGON ((385958.6540616405 762728.3441376068, 385958.6540616405 763728.3441376068, 384958.6540616405 763728.3441376068, 384958.6540616405 762728.3441376068, 385958.6540616405 762728.3441376068))


In [12]:
selected_buildings = buildings[buildings.intersects(user_polygon)]

print("Buildings inside polygon =", len(selected_buildings))

Buildings inside polygon = 1759


In [13]:
# Calculate total rooftop area

roof_area = selected_buildings.geometry.area.sum()

print("Total roof area =", round(roof_area,2), "m²")

Total roof area = 174950.04 m²


In [14]:
solar_irradiance = 1700      # kWh/m²/year
panel_efficiency = 0.20
performance_ratio = 0.75

annual_energy = (
    roof_area
    * solar_irradiance
    * panel_efficiency
    * performance_ratio
)

print("Annual Energy =", round(annual_energy/1e6,2), "GWh/year")

Annual Energy = 44.61 GWh/year


In [15]:
pv_capacity_kw = roof_area / 5

print("Installed PV Capacity =", round(pv_capacity_kw/1000,2), "MW")

Installed PV Capacity = 34.99 MW


In [16]:
number_of_panels = pv_capacity_kw / 0.55

print("Number of 550W panels =", int(number_of_panels))

Number of 550W panels = 63618


In [17]:
average_roof_area = roof_area / len(selected_buildings)

print("Average roof area =", round(average_roof_area,2), "m²/building")

Average roof area = 99.46 m²/building
